# Minor corrections to the TrackPlant3D dataset labels

In [1]:
from pathlib import Path

import numpy as np
import open3d as o3d

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


## Version 1

### 1. Mirrored points and labels in `187_tomato1_control1_plant2_D07.txt`

In [49]:
def rotate_180_y(points):
    """
    Rotate point cloud 180 degrees around Y-axis

    Args:
        points: numpy array of shape (N, 3) where each row is [x, y, z]

    Returns:
        rotated_points: numpy array of same shape with rotated coordinates
    """
    # 180 degree rotation matrix around Y-axis
    # cos(180°) = -1, sin(180°) = 0
    rotation_matrix = np.array([[-1, 0, 0], [0, 1, 0], [0, 0, -1]])

    # Apply rotation: multiply each point by the rotation matrix
    rotated_points = points @ rotation_matrix.T

    print(f"Rotated points by 180° in Y-axis")

    return rotated_points


def rotate_and_swap_labels(
    file_path: str | Path, label_map: dict, output_path: str | Path = None
):
    """Swap labels for wrongly labelled sequences according to label map, e.g. 1->2, 2->1, 3->4, 4->3"""

    points = np.loadtxt(file_path)

    # Rotate points 180 degrees around Y-axis
    points[:, :3] = rotate_180_y(points[:, :3])

    # Swap labels
    labels = points[:, 3].astype(int)
    new_labels = np.array([label_map.get(label, label) for label in labels])
    points[:, 3] = new_labels

    # Save to output path or overwrite original
    save_path = output_path if output_path else file_path
    np.savetxt(save_path, points, fmt="%f %f %f %d")
    print(f"Label changes: 1↔2, 3↔4")
    print(f"Saved file to: {save_path}")

    return points


##### Wrongly labelled scan 187_tomato1_control1_plant2_D07.txt #####
# Rotate 180° in Y-axis then swap labels (verified in CloudCompare that this would be the correct orientation and labels)
file_path = "../data/TrackPlant3D/gt/tomato/187_tomato1_control1_plant2_D07.txt"
output_path = (
    "../data/TrackPlant3D/corrections/gt_edited/tomato/187_tomato1_control1_plant2_D07.txt"
)
if not Path(output_path).parent.exists():
    Path(output_path).parent.mkdir(parents=True)

label_map = {1: 2, 2: 1, 3: 4, 4: 3}
rotate_and_swap_labels(file_path, label_map=label_map, output_path=output_path)


### 2. Few wrongly labeled points in scan `231_tomato1_drought_plant3_D08`

The following points are labelled as stem (0) in leaf 1 but they're in the middle so they should clearly be leaf 1:

In [50]:
# Got these visually in CloudCompare
file = "../data/TrackPlant3D/corrections/points_lists/231_tomato1_drought_plant3_D08_points_to_relabel_as_1.txt"
np.sort(np.loadtxt(file, delimiter=",")[...,0].astype(int))

In [51]:
indices = np.array([15, 16, 17, 25, 26, 37, 55, 59, 60, 61, 62, 75, 76, 106, 107, 109, 110, 116,
                    117, 129, 130, 158, 159, 162, 167, 168, 179, 216, 217, 218, 221, 222, 230, 231, 282, 283,
                    288, 289])

In [52]:
input_path = "../data/TrackPlant3D/gt/tomato/231_tomato1_drought_plant3_D08.txt"
output_path = "../data/TrackPlant3D/corrections/gt_edited/tomato/231_tomato1_drought_plant3_D08.txt"
if not Path(output_path).parent.exists():
    Path(output_path).parent.mkdir(parents=True)

In [53]:
points = np.loadtxt(input_path)
points[indices, 3] = 1

np.savetxt(output_path, points, fmt="%f %f %f %d")
print(f"Changed {len(indices)} points from label 0 to label 1")
print(f"Saved corrected file to: {output_path}")

### 3. Some sparse points in stem labeled as leaves in `312_tomato1_shade_plant1_D06.txt`

In [54]:
## Points to relabel picked by visual inspection in CloudCompare
file = "../data/TrackPlant3D/corrections/points_lists/312_tomato1_shade_plant1_D06_points_to_relabel_as_0.txt"
np.sort(np.loadtxt(file, delimiter=",")[...,0].astype(int))

In [55]:
indices = np.array([ 733,  736,  933,  937,  941,  943,  945, 1279, 1284, 1286, 1692,
       1698, 1706, 1710, 2200, 2218, 2219, 2220, 2221, 2732, 2738, 2739,
       2740, 2742, 2744, 2745, 3255, 3256, 3591, 3592, 3593, 3594, 3595,
       3596, 3597, 3598, 3599, 3600, 3602, 3603, 3604, 3605, 3606, 3607,
       3876, 3877, 4112])

In [56]:
input_path = "../data/TrackPlant3D/gt/tomato/312_tomato1_shade_plant1_D06.txt"
output_path = "../data/TrackPlant3D/corrections/gt_edited/tomato/312_tomato1_shade_plant1_D06.txt"
if not Path(output_path).parent.exists():
    Path(output_path).parent.mkdir(parents=True)

In [57]:
points = np.loadtxt(input_path)

# Relabel selected points as stem (0)
points[indices, 3] = 0

# Save the modified points
np.savetxt(output_path, points, fmt="%f %f %f %d")
print(f"Changed {len(indices)} points from label 0 to label 1")
print(f"Saved corrected file to: {output_path}")

### 4. Scan `128_tobacco_control_plant3_D08` leaf label 1 has two leaves

There is leaf 6 (if looking at the next scan) which is already clearly visible in this scan but still labeled as leaf 1. This is a mistake in the original dataset, so we can easily correct to gain an extra leaf label

In [58]:
## Points to relabel obtained from visual inspection in CloudCompare
file = "../data/TrackPlant3D/corrections/points_lists/128_tobacco_control_plant3_D08_relabel_as_6.txt"
np.sort(np.loadtxt(file, delimiter=",")[...,0].astype(int))

In [59]:
indices = np.array([2666, 2668, 2669, 2671, 2672, 2675, 2808, 2814, 2828, 2829, 2830,
       2831, 2832, 2833, 2834, 2835, 2974, 2975, 2977, 2978, 2980, 2981,
       2984, 2985, 2986, 2987, 2988, 2989, 2990, 2991, 2992, 2993, 2994,
       3112, 3116, 3117, 3118, 3121, 3122, 3123, 3126, 3127, 3129, 3130,
       3131, 3132, 3133, 3134, 3135, 3136, 3137, 3138, 3139, 3140, 3252,
       3257, 3258, 3259, 3260, 3262, 3263, 3264, 3267, 3268, 3269, 3271,
       3272, 3273, 3277, 3278, 3279, 3282, 3283, 3285, 3286, 3287, 3288,
       3289, 3290, 3408, 3411, 3412, 3415, 3416, 3418, 3419, 3420, 3422,
       3423, 3424, 3427, 3428, 3431, 3433, 3435, 3548, 3551, 3555, 3558])

In [60]:
filename = "128_tobacco_control_plant3_D08.txt"
input_path = f"../data/TrackPlant3D/gt/tobacco/{filename}"
output_path = f"../data/TrackPlant3D/corrections/gt_edited/tobacco/{filename}"
if not Path(output_path).parent.exists():
    Path(output_path).parent.mkdir(parents=True)

In [61]:
points = np.loadtxt(input_path)

# Relabel selected points
points[indices, 3] = 6

# Save the modified points
np.savetxt(output_path, points, fmt="%f %f %f %d")
print(f"Changed {len(indices)} points from label 0 to label 6")
print(f"Saved corrected file to: {output_path}")

### 5. Scan `134_tobacco_heat_plant2_D04` leaf 5 noisy points in peduncle of another leaf

In [62]:
## Points to relabel obtained from visual inspection in CloudCompare
file = "../data/TrackPlant3D/corrections/points_lists/134_tobacco_heat_plant2_D04_relabel_as_0.txt"
np.sort(np.loadtxt(file, delimiter=",")[...,0].astype(int))

In [63]:
indices = np.array([3224, 3364, 3365, 3366, 3367, 3370, 3506, 3507, 3508, 3509, 3510,
       3511, 3512, 3513, 3514, 3515, 3639, 3640, 3641, 3642, 3643, 3644,
       3645, 3646, 3647, 3785, 3786, 3787, 3788, 3789, 3790, 3791, 3792,
       3794, 3972, 3974, 3976, 3977, 3979, 3980, 3981, 3982, 3983, 3984,
       4224, 4225, 4227, 4235, 4241, 4248, 4466, 4471, 4477, 4673, 4676,
       5669, 5672, 5673, 5757, 5760])

In [64]:
filename = "134_tobacco_heat_plant2_D04.txt"
input_path = f"../data/TrackPlant3D/gt/tobacco/{filename}"
output_path = f"../data/TrackPlant3D/corrections/gt_edited/tobacco/{filename}"
if not Path(output_path).parent.exists():
    Path(output_path).parent.mkdir(parents=True)

In [65]:
points = np.loadtxt(input_path)

# Relabel selected points
points[indices, 3] = 0

# Save the modified points
np.savetxt(output_path, points, fmt="%f %f %f %d")
print(f"Changed {len(indices)} points to label 0")
print(f"Saved corrected file to: {output_path}")

### 6. `141_tobacco_heat_plant3_D06.txt` in leaf 6 is currently labelled as stem (0)

In [66]:
## Points to relabel obtained from visual inspection in CloudCompare
file = "../data/TrackPlant3D/corrections/points_lists/141_tobacco_heat_plant3_D06_relabel_as_6.txt"
np.loadtxt(file, delimiter=",")[..., 0].astype(int)

In [67]:
indices = np.array([6800])

In [68]:
filename = "141_tobacco_heat_plant3_D06.txt"
input_path = f"../data/TrackPlant3D/gt/tobacco/{filename}"
output_path = f"../data/TrackPlant3D/corrections/gt_edited/tobacco/{filename}"
if not Path(output_path).parent.exists():
    Path(output_path).parent.mkdir(parents=True)

In [69]:
points = np.loadtxt(input_path)

# Relabel selected points
points[indices, 3] = 6

# Save the modified points
np.savetxt(output_path, points, fmt="%f %f %f %d")
print(f"Changed {len(indices)} points from label 5 to label 0")
print(f"Saved corrected file to: {output_path}")

## Version 2

### 1. Scan `tobacco_shade_plant3` labelling errors

Symmetric labelling issue from timestep 3 onwards. Need to swap 1 with 2, and 3 with 4.

In [ ]:
# Helper function
def swap_labels(file_path: str | Path, label_map: dict, output_path: str | Path = None):
    """Swap labels for wrongly labelled sequences according to label map, e.g. 1->2, 2->1, 3->4, 4->3"""

    points = np.loadtxt(file_path)

    # Swap labels
    labels = points[:, 3].astype(int)
    new_labels = np.array([label_map.get(label, label) for label in labels])
    points[:, 3] = new_labels

    # Save to output path or overwrite original
    save_path = output_path if output_path else file_path
    np.savetxt(save_path, points, fmt="%f %f %f %d")
    print(f"Label changes: 1↔2, 3↔4")
    print(f"Saved file to: {save_path}")

    return points

In [ ]:
# Define paths
base_data_path = Path("../data/TrackPlant3D/gt_corrected_v1/tobacco")
base_output_path = Path("../data/TrackPlant3D/corrections/v2/gt_edited_v1/tobacco")

assert base_data_path.exists(), f"Base data path does not exist: {base_data_path}"

if not base_output_path.exists():
    base_output_path.mkdir(parents=True)

#### 1.1. D06 (timestep 3) swaped labels

In [ ]:
file_name = "162_tobacco_shade_plant3_D06.txt"
label_map = {1: 2, 2: 1, 3: 4, 4: 3}

swap_labels(base_data_path / file_name, label_map, base_output_path / file_name)

#### 1.2. D08 (timestep 4) swaped labels

In [ ]:
file_name = "163_tobacco_shade_plant3_D08.txt"
label_map = {1: 2, 2: 1, 3: 4, 4: 3}

swap_labels(base_data_path / file_name, label_map, base_output_path / file_name)

#### 1.3. D10 (timestep 5) swaped labels

In [ ]:
file_name = "164_tobacco_shade_plant3_D10.txt"
label_map = {1: 2, 2: 1, 3: 4, 4: 3}

swap_labels(base_data_path / file_name, label_map, base_output_path / file_name)

#### 1.4. D12 (timestep 6) swaped labels

In [ ]:
file_name = "165_tobacco_shade_plant3_D12.txt"
label_map = {1: 2, 2: 1, 3: 4, 4: 3}

swap_labels(base_data_path / file_name, label_map, base_output_path / file_name)

#### 1.5. D14 (timestep 8) swaped labels

In [ ]:
file_name = "166_tobacco_shade_plant3_D14.txt"
label_map = {1: 2, 2: 1, 3: 4, 4: 3}

swap_labels(base_data_path / file_name, label_map, base_output_path / file_name)

Symmetric labelling issue from timestep 3 onwards. Need to swap 1 with 2, and 3 with 4.

In [9]:
# Helper function
def swap_labels(file_path: str | Path, label_map: dict, output_path: str | Path = None):
    """Swap labels for wrongly labelled sequences according to label map, e.g. 1->2, 2->1, 3->4, 4->3"""

    points = np.loadtxt(file_path)

    # Swap labels
    labels = points[:, 3].astype(int)
    new_labels = np.array([label_map.get(label, label) for label in labels])
    points[:, 3] = new_labels

    # Save to output path or overwrite original
    save_path = output_path if output_path else file_path
    np.savetxt(save_path, points, fmt="%f %f %f %d")
    print(f"Label changes: 1↔2, 3↔4")
    print(f"Saved file to: {save_path}")

    return points

In [17]:
# Define paths
base_data_path = Path("../data/TrackPlant3D/gt_corrected_v1/tobacco")
base_output_path = Path("../data/TrackPlant3D/corrections/v2/gt_edited_v1/tobacco")

assert base_data_path.exists(), f"Base data path does not exist: {base_data_path}"

if not base_output_path.exists():
    base_output_path.mkdir(parents=True)

#### 1.1. D06 (timestep 3) swaped labels

In [18]:
file_name = "162_tobacco_shade_plant3_D06.txt"
label_map = {1: 2, 2: 1, 3: 4, 4: 3}

swap_labels(base_data_path / file_name, label_map, base_output_path / file_name)

#### 1.2. D08 (timestep 4) swaped labels

In [22]:
file_name = "163_tobacco_shade_plant3_D08.txt"
label_map = {1: 2, 2: 1, 3: 4, 4: 3}

swap_labels(base_data_path / file_name, label_map, base_output_path / file_name)

#### 1.3. D10 (timestep 5) swaped labels

In [23]:
file_name = "164_tobacco_shade_plant3_D10.txt"
label_map = {1: 2, 2: 1, 3: 4, 4: 3}

swap_labels(base_data_path / file_name, label_map, base_output_path / file_name)

#### 1.4. D12 (timestep 6) swaped labels

In [24]:
file_name = "165_tobacco_shade_plant3_D12.txt"
label_map = {1: 2, 2: 1, 3: 4, 4: 3}

swap_labels(base_data_path / file_name, label_map, base_output_path / file_name)

#### 1.5. D14 (timestep 8) swaped labels

In [25]:
file_name = "166_tobacco_shade_plant3_D14.txt"
label_map = {1: 2, 2: 1, 3: 4, 4: 3}

swap_labels(base_data_path / file_name, label_map, base_output_path / file_name)

###

### Write version

In [33]:
!cp -r ../data/TrackPlant3D/gt_corrected_v1 ../data/TrackPlant3D/gt_corrected_v2

In [34]:
!rsync -avr ../data/TrackPlant3D/corrections/v2/gt_edited_v1/ ../data/TrackPlant3D/gt_corrected_v2/

## Version 2 (again)

### 1. Scan `tobacco_shade_plant3` labelling errors

Here we post correct version 2 ply files (which has corrected normals and aligned plants) because I forgot to generate it from the gt_corrected_v2 corrected labels we did above with tht txt files. We apply this changes again and potentiall other small fixes.

In [3]:
def swap_labels_ply(file_path: Path, label_map: dict, output_path: Path = None):
    """
    Swap labels in PLY file according to label map
    Preserves all other attributes: positions, normals, is_leaf_tip

    Args:
        file_path: Path to input PLY file
        label_map: Dict mapping old labels to new labels (e.g., {1: 2, 2: 1, 3: 4, 4: 3})
        output_path: Path to save corrected PLY (if None, overwrites input)
    """
    # Load PLY file (loads all attributes: positions, normals, organ_label, is_leaf_tip)
    pcd = o3d.t.io.read_point_cloud(str(file_path))

    # Get current labels
    labels = pcd.point.organ_label.numpy().flatten().astype(int)

    # Count changes for verification
    unique_before = np.unique(labels, return_counts=True)

    # Swap labels according to map
    new_labels = np.array([label_map.get(label, label) for label in labels])

    # Count after
    unique_after = np.unique(new_labels, return_counts=True)

    # Update labels in point cloud (keeps positions, normals, is_leaf_tip unchanged)
    pcd.point.organ_label = o3d.core.Tensor(
        new_labels.reshape(-1, 1), dtype=o3d.core.Dtype.Int32
    )

    # Verify other attributes are still present
    assert hasattr(pcd.point, 'positions'), "positions lost!"
    assert hasattr(pcd.point, 'normals'), "normals lost!"
    assert hasattr(pcd.point, 'is_leaf_tip'), "is_leaf_tip lost!"

    # Save (writes all attributes)
    save_path = output_path if output_path else file_path
    o3d.t.io.write_point_cloud(str(save_path), pcd)

    print(f"  Labels before: {dict(zip(unique_before[0], unique_before[1]))}")
    print(f"  Labels after:  {dict(zip(unique_after[0], unique_after[1]))}")
    print(f"  ✓ Saved with normals and leaf tips preserved")

In [2]:
# Define paths
base_ply_path = Path("../data/TrackPlant3D/versions/v2/gt_corrected_v2/tobacco")
assert base_ply_path.exists(), f"Base PLY path does not exist: {base_ply_path}"

In [4]:
# Files to correct (tobacco_shade_plant3 from timestep 3 onwards)
files_to_correct = [
    "162_tobacco_shade_plant3_D06.ply",  # timestep 3
    "163_tobacco_shade_plant3_D08.ply",  # timestep 4
    "164_tobacco_shade_plant3_D10.ply",  # timestep 5
    "165_tobacco_shade_plant3_D12.ply",  # timestep 6
    "166_tobacco_shade_plant3_D14.ply",  # timestep 7
]

In [5]:
# Label swap: 1↔2, 3↔4
label_map = {1: 2, 2: 1, 3: 4, 4: 3}

In [6]:
# Apply corrections
for file_name in files_to_correct:
    file_path = base_ply_path / file_name
    if file_path.exists():
        print(f"\nProcessing: {file_name}")
        swap_labels_ply(file_path, label_map, output_path=file_path)
    else:
        print(f"Warning: File not found: {file_path}")

print("\n✓ All label corrections applied to v2 PLY files!")

### 2. Delete leaf tip for `sorghum_highlight_plant2` leaf 1 at D10

First, let's visualize leaf 1 to confirm it's the right leaf that needs correction:

In [6]:
# Load the PLY file
base_ply_path = Path("../data/TrackPlant3D/versions/v2/gt_corrected_v2/sorghum/")
filename = "79_sorghum_highlight_plant2_D10.ply"
file_path = base_ply_path / filename

# Load PLY file
pcd = o3d.t.io.read_point_cloud(str(file_path))

# Get all data
positions = pcd.point.positions.numpy()
labels = pcd.point.organ_label.numpy().flatten()
normals = pcd.point.normals.numpy()
is_leaf_tip = pcd.point.is_leaf_tip.numpy().flatten()

# Filter for leaf 1 only
leaf_1_mask = labels == 1
leaf_1_positions = positions[leaf_1_mask]
leaf_1_normals = normals[leaf_1_mask]
leaf_1_is_tip = is_leaf_tip[leaf_1_mask]

print(f"Total points in scan: {len(positions)}")
print(f"Points in leaf 1: {len(leaf_1_positions)}")
print(f"Leaf 1 tip points: {leaf_1_is_tip.sum()}")
print(f"Leaf 1 indices in full cloud: {np.where(leaf_1_mask)[0][:20]}...")  # Show first 20

# Create point cloud for visualization
vis_pcd = o3d.geometry.PointCloud()
vis_pcd.points = o3d.utility.Vector3dVector(leaf_1_positions)
vis_pcd.normals = o3d.utility.Vector3dVector(leaf_1_normals)

# Color by is_leaf_tip: tip points in RED, others in BLUE
colors = np.zeros((len(leaf_1_positions), 3))
colors[leaf_1_is_tip == 1] = [1, 0, 0]  # Red for tip
colors[leaf_1_is_tip == 0] = [0, 0, 1]  # Blue for non-tip
vis_pcd.colors = o3d.utility.Vector3dVector(colors)

print(f"\nLeaf 1 Z-range: [{leaf_1_positions[:, 2].min():.2f}, {leaf_1_positions[:, 2].max():.2f}]")

# Visualize
o3d.visualization.draw_geometries([vis_pcd], window_name="Leaf 1 - sorghum_highlight_plant2_D10 (RED=tip, BLUE=non-tip)")

Total points in scan: 8598
Points in leaf 1: 995
Leaf 1 tip points: 0
Leaf 1 indices in full cloud: [5088 5207 5208 5209 5210 5211 5212 5213 5214 5215 5216 5217 5218 5310
 5311 5313 5314 5315 5316 5317]...

Leaf 1 Z-range: [4.14, 38.94]


Now let's remove the leaf tip attribute for leaf 1 (set is_leaf_tip = 0) and save the corrected file:

In [4]:
# Reload the file to make the correction
pcd = o3d.t.io.read_point_cloud(str(file_path))

# Get all data
labels = pcd.point.organ_label.numpy().flatten()
is_leaf_tip = pcd.point.is_leaf_tip.numpy().flatten().copy()

# Find indices where it's leaf 1 AND marked as leaf tip
leaf_1_tip_mask = (labels == 1) & (is_leaf_tip == 1)
num_points_to_modify = leaf_1_tip_mask.sum()
print(f"Points to modify (leaf 1 tips): {num_points_to_modify}")
print(f"Total points: {len(labels)}")
print(f"Leaf 1 tip points before: {is_leaf_tip[labels == 1].sum()}")

Points to modify (leaf 1 tips): 1
Total points: 8598
Leaf 1 tip points before: 1


In [5]:
# Set is_leaf_tip to 0 for all leaf 1 points that are currently marked as tip
is_leaf_tip[leaf_1_tip_mask] = 0

# Update the point cloud with modified is_leaf_tip
pcd.point.is_leaf_tip = o3d.core.Tensor(
    is_leaf_tip.reshape(-1, 1), dtype=o3d.core.Dtype.Int32
)

print(f"Leaf 1 tip points after: {pcd.point.is_leaf_tip.numpy().flatten()[labels == 1].sum()}")

# Verify other attributes are unchanged
assert hasattr(pcd.point, 'positions'), "positions lost!"
assert hasattr(pcd.point, 'normals'), "normals lost!"
assert hasattr(pcd.point, 'organ_label'), "organ_label lost!"

# Save corrected file (overwrite in place)
o3d.t.io.write_point_cloud(str(file_path), pcd)
print(f"\n✓ Saved corrected file to: {file_path}")
print(f"✓ Modified {num_points_to_modify} points (set is_leaf_tip = 0 for leaf 1)")

Leaf 1 tip points after: 0

✓ Saved corrected file to: ../data/TrackPlant3D/versions/v2/gt_corrected_v2/sorghum/79_sorghum_highlight_plant2_D10.ply
✓ Modified 1 points (set is_leaf_tip = 0 for leaf 1)
